In [0]:
%run ../init_framework

In [0]:
# # --- 1. Initialize the CDF Read Stream for Defaulters ---
# df_bronze_defaulters = (spark.readStream
#     .format("delta")
#     .option("readChangeFeed", "true")
#     .option("startingVersion", 1) 
#     .table(DEFAULTERS_BRONZE))

In [0]:
# --- 1. Initialize the CDF Read Stream for Defaulters ---
df_bronze_defaulters = (spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1) 
    .table(DEFAULTERS_BRONZE))

In [0]:
df_with_metadata = apply_silver_metadata(df_bronze_defaulters)

In [0]:
df_distinct = df_with_metadata.distinct()

In [0]:
from pyspark.sql.functions import col

# 1. Cast to float first to avoid losing decimal strings to NULL
# 2. Cast to int to perform the "floor" truncation
# 3. Replace all NULLs (original nulls + invalid strings) with 0

df_delinq2yrs_fixed = df_distinct.withColumn(
    "delinq_2yrs", col("delinq_2yrs").cast("float").cast("int")
).fillna(0, subset=["delinq_2yrs"])

In [0]:
from pyspark.sql.functions import col

df_defaulters_delinq = (df_delinq2yrs_fixed
    .select(
        col("member_id"),
        col("delinq_2yrs"),
        col("delinq_amnt"),
        col("mths_since_last_delinq").cast("int"),
        # Metadata Columns
        col("_ingested_at"),
        col("_source_file"),
        col("_bronze_version"),
        col("_updated_at")
    )
    .filter((col("delinq_2yrs") > 0) | (col("mths_since_last_delinq") > 0))
)

In [0]:
from pyspark.sql.functions import col

# Direct DataFrame API implementation for Enquiries with Metadata
df_defaulters_inquiry = (df_delinq2yrs_fixed
    .select(
        col("member_id"), 
        col("pub_rec"), 
        col("pub_rec_bankruptcies"), 
        col("inq_last_6mths"),
        # Metadata Columns included in projection
        col("_ingested_at"),
        col("_source_file"),
        col("_bronze_version"),
        col("_updated_at")
    )
    .filter(
        (col("pub_rec") > 0.0) | 
        (col("pub_rec_bankruptcies") > 0.0) | 
        (col("inq_last_6mths") > 0.0)
    )
)

In [0]:
df_defaulters_inquiry.limit(3).display()